[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AGRIdigital-Climbat/PlantillaMapa/blob/main/01_Fisiografia/01_fisiografia.ipynb)

# Fisiografia: relieve, pendiente y drenaje del municipio
## PROSUCO - CLIMBAT | Analisis por municipio de Bolivia (GEE + Colab)

> **Area de estudio = tu GeoJSON.** Este notebook usa por defecto:
>
> `RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"` (compartido `Cualquiera con el enlace`).
> El GeoJSON **siempre manda**: al cargarse, `roi`, `ZONA['bbox']`, `ZONA['centro']`
> y `area_km2` se recalculan desde el poligono. El `bbox` de CONFIG solo es respaldo
> si cambias a `AOI_MODE = "bbox"`.

**Orden de ejecucion:** 1) Instalacion -> 2) Autenticacion GEE (`EE_PROJECT`) ->
3) CONFIG (anos + satelite + umbrales) -> 4) AOI desde Drive -> 5) celdas del modulo.
Solo edita la celda **CONFIGURACION MAESTRA**.


In [ ]:
# ============================================================
# 1.1 Instalación de librerías (ejecutar una sola vez en Colab)
# ============================================================

import importlib
import subprocess
import sys


def _asegurar(paquete, modulo=None):
    """Instala el paquete solo si no está disponible."""
    try:
        importlib.import_module(modulo or paquete)
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", paquete]
        )


_asegurar("earthengine-api", "ee")
_asegurar("geemap")
_asegurar("geopandas")
_asegurar("pandas")
_asegurar("numpy")
_asegurar("matplotlib")

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("Librerías listas.")
print("earthengine-api:", ee.__version__)
print("geemap:", geemap.__version__)

In [ ]:
# ============================================================
# 1.2 Autenticación en Google Earth Engine
# ============================================================
# >>> REEMPLAZAR por el id del proyecto de Cloud de PROSUCO/CLIMBAT <<<

EE_PROJECT = "iotapp7"

ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

print("Earth Engine inicializado correctamente.")
print("Proyecto:", EE_PROJECT)

## Configuración maestra

> **✏️ Esta es la única celda que se edita para diagnosticar una zona nueva.**

| Bloque | Qué define | Verificación previa |
|---|---|---|
| `ZONA` | nombre, bbox `[oeste, sur, este, norte]` y punto de vista | dibujar en [geojson.io](https://geojson.io) o en el editor GEE y copiar coordenadas |
| `AOI_MODE`/`RUTA_AOI` | fuente del AOI: `"bbox"` o `"auto"` (polígono **GeoJSON**) | exportar el polígono desde QGIS como **GeoJSON**; en `RUTA_AOI` se puede pegar un **enlace de Google Drive** (el notebook lo descarga solo) o una ruta de Drive |
| `PUNTOS_LOCALES` | manantiales/bofedales del saber local | levantamiento con las comunidades |
| `FECHA_INICIO / FECHA_FIN` | ventana analítica (**30+ años**) | default 1996-01-01 → 2026-12-31 |
| `CALENDARIO` | meses de siembra/cosecha y estación seca | observadores Yapuchiri |
| `UMBRALES` | todos los cortes analíticos en un solo lugar | criterio técnico + experiencia local |
| `ESCALAS` | resolución de cómputo (m); subir si la zona es grande | — |

In [ ]:
# ===========================================================================
# CONFIGURACIÓN MAESTRA — ÚNICA CELDA QUE SE EDITA PARA CADA ZONA
# ===========================================================================

# --- 1. Zona de intervención ------------------------------------------------
# Fuente del AOI:
#   "bbox" -> usa el rectángulo ZONA["bbox"] de abajo (rápido, sin Drive)
#   "auto" -> usa un polígono .geojson. RUTA_AOI admite 3 formas:
#       * Enlace o ID de Google Drive  -> se DESCARGA solo (no subes nada a mano)
#           "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"
#           "11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N"
#       * Ruta en tu Drive ya montado  -> se lee directo, SIN copiar a /content
#           "/content/drive/MyDrive/CLIMBAT/zona.geojson"
#       * Carpeta en Drive             -> toma el único .geojson que contenga
#           "/content/drive/MyDrive/CLIMBAT"
AOI_MODE = "auto"
RUTA_AOI = "https://drive.google.com/file/d/11k-2-0MfG84QcYld7XY3r5XVMDHWtJ0N/view?usp=sharing"

ZONA = {
    # Coordenadas geocodificadas: Escoma (La Paz, Bolivia) ~3832 msnm
    # (solo se usan como respaldo cuando AOI_MODE = "bbox")
    "nombre": "Escoma, La Paz (prueba)",
    "bbox": [-69.33, -15.86, -68.93, -15.46],   # [oeste, sur, este, norte]
    "centro": [-69.1289, -15.6615],             # punto de vista del mapa
    "zoom": 10,
}

# --- 2. Puntos levantados con comunidades (saber local) ---------------------
# Listas de [longitud, latitud]. Dejar vacías si aún no hay levantamiento.
PUNTOS_LOCALES = {
    "manantiales": [],
    "bofedales": [],
}

# --- 3. Ventana analítica: 30+ años hacia atrás desde el año actual ---------
# Regla de la plantilla: 1996-01-01 -> 2026-12-31 (todo análisis de rango de años usa esta).
# Los datasets usan su período nativo si es mayor: CHIRPS 1981+ · ERA5-Land 1950+
# · JRC 1984–2021 · MODIS 2000+ (NDVI) / 2001+ (MCD12Q1 agricultura) · SMAP 2015+
FECHA_INICIO = "1996-01-01"
FECHA_FIN = "2026-12-31"

# --- 4. Calendario agrícola local (verificar con observadores Yapuchiri) ----
CALENDARIO = {
    "siembra": 11,       # mes de siembra (1–12)
    "cosecha": 4,        # mes de cosecha (1–12)
    "meses_estacion_seca": [5, 6, 7, 8, 9, 10],   # se usa en M10 (bofedales)
}

# --- 5. Umbrales analíticos (ajustar con criterio técnico y local) ----------
UMBRALES = {
    "helada_c": 0.0,               # °C mínima para contar un día de helada (M4)
    "helada_grave_c": -2.0,        # °C de helada grave (daño severo) (M4)
    "mes_seco_mm": 30.0,           # mm/mes para clasificar mes seco (M2)
    "spi_severo": -1.5,            # umbral de sequía severa SPI-3 (M5)
    "ndvi_anomalia_z": -1.5,       # anomalía NDVI severa (M7)
    "manantial_occurrencia": 100,  # % ocurrencia JRC mínima (M9)
    "manantial_drenaje_max": 50,   # celdas acumuladas máx. (cabeceras) (M9)
    "bofedal_altitud_min": 3000,   # msnm mínimo (M10)
    "bofedal_altitud_max": 5200,   # msnm máximo (M10)
    "bofedal_pendiente_max": 5.0,  # grados de pendiente máxima (M10)
    "bofedal_seasonality": 8,      # meses con agua JRC mínimos (M10)
    "bofedal_ndvi_seco": 0.25,     # NDVI mínimo en estación seca (M10)
    "clases_agricolas": [12, 14],  # LC_Type1 MCD12Q1: 12=cultivos, 14=mosaico (M11)
}

# --- 6. Escalas de cómputo (metros) -----------------------------------------
# Subir si la zona es muy grande (más rápido, menos detalle)
ESCALAS = {"clima": 5000, "modis": 250, "jrc": 90, "detalle": 30}
TILE_SCALE = 4

# ===========================================================================
# A partir de aquí no se edita nada
# ===========================================================================

MESES = list(range(1, 13))
MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun",
            "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]

roi = None   # se construye en la celda 1.4 (bbox o GeoJSON según AOI_MODE)
centro = None
ANIO_INI = int(FECHA_INICIO[:4])
ANIO_FIN = int(FECHA_FIN[:4])

# Años COMPLETOS (se excluye el año en curso de métricas y mapas anuales)
ANIOS_COMPLETOS = list(range(ANIO_INI, ANIO_FIN))
FILTRO_ANIOS_COMPLETOS = ee.Filter.calendarRange(
    ANIOS_COMPLETOS[0], ANIOS_COMPLETOS[-1], "year"
)

print("Zona:", ZONA["nombre"])
print("Ventana analítica:", FECHA_INICIO, "->", FECHA_FIN,
      f"({ANIO_FIN - ANIO_INI + 1} años calendario)")
print("Años completos para métricas:", ANIOS_COMPLETOS[0], "-", ANIOS_COMPLETOS[-1])
print("Calendario: siembra mes", CALENDARIO["siembra"],
      "| cosecha mes", CALENDARIO["cosecha"])
print("Umbrales activos:", UMBRALES)

In [ ]:
# ============================================================
# 1.3 Utilidades comunes (reutilizadas por todos los módulos)
# ============================================================

# Parámetros de reducción seguros contra "User memory limit exceeded":
#   bestEffort=True -> GEE sube la escala automáticamente si no cabe en memoria.
#   tileScale=8     -> divide el cálculo en tiles más pequeños (menos memoria c/u).
REDUCIR = dict(maxPixels=1e9, tileScale=8, bestEffort=True)
REDUCIR_AREA = dict(maxPixels=1e10, tileScale=16, bestEffort=True)


def apilar(imagenes, nombres):
    """Une imágenes de una banda en una sola imagen multibanda."""
    pila = None
    for img, nombre in zip(imagenes, nombres):
        banda = ee.Image(img).select(0).rename(nombre)
        pila = banda if pila is None else pila.addBands(banda)
    return pila


def reducir_bandas(imagenes, nombres, escala, tam=6, reductor=None, params=None):
    """reduceRegion de muchas bandas EN BLOQUES (una petición por bloque).

    Evita el error de memoria que ocurre al reducir de una sola vez una imagen
    con decenas de bandas (cada banda puede depender de cientos/miles de imágenes
    diarias). Devuelve un dict {nombre_banda: valor}.
    """
    reductor = reductor or ee.Reducer.mean()
    params = params or REDUCIR
    salida = {}
    for i in range(0, len(imagenes), tam):
        pila = apilar(imagenes[i:i + tam], nombres[i:i + tam])
        salida.update(
            pila.reduceRegion(reductor, roi, escala, **params).getInfo()
        )
    return salida


def composite_mensual(coleccion, anios, banda=None):
    """Reduce una colección diaria a UN compuesto por año-mes (una imagen c/u).

    Aligera ~30x la memoria de las climatologías de 30 años. `anios` debe listar
    solo años con dato (evita colecciones vacías). Con `banda`, deja una sola banda.
    """
    imgs = []
    for y in anios:
        for m in MESES:
            i0 = f"{y}-{m:02d}-01"
            i1 = (pd.Timestamp(i0) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")
            sub = coleccion.filterDate(i0, i1)
            if banda:
                sub = sub.select(banda)
            imgs.append(sub.mean().set("system:time_start", ee.Date(i0).millis()))
    return ee.ImageCollection(imgs)


def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente).

    Evita que el relleno de color tape el mapa y meta ruido en la interpretación;
    deja el límite del área bien visible por encima de las demás capas.
    """
    return roi.style(color=color, fillColor="00000000", width=ancho)


def promediar_aoi(imagen, banda, escala):
    """Promedio de una banda dentro del AOI -> valor Python (o None)."""
    d = (
        imagen.select(banda)
        .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        .getInfo()
    )
    return d.get(banda)


def serie_aoi(coleccion, banda, escala, bloque_anios=5):
    """Extrae el promedio del AOI por imagen, EN BLOQUES de años.

    Procesar la serie completa de una vez agota la memoria; por eso se trocea
    (por defecto cada 5 años) y se concatenan los resultados.
    """
    registros = []

    def _una_imagen(img):
        st = (
            img.select(banda)
            .reduceRegion(ee.Reducer.mean(), roi, escala, **REDUCIR)
        )
        return ee.Feature(None, {
            "date": ee.Date(img.get("system:time_start")).format("yyyy-MM-dd"),
            "valor": st.get(banda),
        })

    for y0 in range(ANIO_INI, ANIO_FIN + 1, bloque_anios):
        y1 = min(y0 + bloque_anios - 1, ANIO_FIN)
        sub = coleccion.filterDate(f"{y0}-01-01", f"{y1 + 1}-01-01")
        fc = ee.FeatureCollection(sub.map(_una_imagen)).getInfo()
        registros += [f["properties"] for f in fc["features"]]
    return registros


print("Utilidades: apilar(), reducir_bandas(), composite_mensual(), "
      "promediar_aoi(), serie_aoi()")

In [ ]:
# ============================================================
# 1.4 Área de estudio (AOI) y mapa base
# ============================================================
# Fuente del AOI (ver CONFIG):
#   AOI_MODE = "bbox" -> rectángulo ZONA["bbox"].
#   AOI_MODE = "auto" -> polígono desde un archivo .geojson en Google Drive.
#                        (En QGIS: Exportar -> Guardar entidades como -> GeoJSON,
#                         CRS EPSG:4326; subir a Drive y ajustar RUTA_AOI.)

def borde_aoi(color="FF0000", ancho=2):
    """Capa de SOLO el borde del AOI (relleno transparente)."""
    # ee.Geometry NO tiene .style() en la API de Python -> se envuelve en FeatureCollection
    fc = ee.FeatureCollection([ee.Feature(roi)])
    return fc.style(color=color, fillColor="00000000", width=ancho)


if AOI_MODE == "auto":
    from google.colab import drive
    drive.mount("/content/drive")

    import os
    import re
    import glob
    import geopandas as gpd

    ruta = str(RUTA_AOI).strip()

    # (a) Enlace o ID de Google Drive -> descarga automática (sin subir nada a mano).
    #     El archivo debe estar compartido como "Cualquiera con el enlace".
    if ruta.startswith("http") or ("/" not in ruta and len(ruta) >= 20):
        _asegurar("gdown")
        import gdown

        m = re.search(r"(?:/d/|id=)([-\w]{20,})", ruta)
        fid = m.group(1) if m else ruta
        ext = (
            ".zip" if ruta.lower().endswith(".zip") else
            ".kml" if ruta.lower().endswith(".kml") else
            ".gpkg" if ruta.lower().endswith(".gpkg") else
            ".geojson"
        )
        destino = "/content/zona_aoi" + ext
        print("Descargando AOI desde Google Drive...")
        gdown.download(id=fid, output=destino, quiet=False)
        ruta = destino
    # (b) Carpeta en Drive -> usar el único .geojson que contenga
    elif os.path.isdir(ruta):
        cands = sorted(glob.glob(os.path.join(ruta, "*.geojson")))
        if not cands:
            raise FileNotFoundError(f"No se encontró ningún .geojson en {ruta}")
        ruta = cands[0]
    # (c) Ruta de archivo en el Drive montado -> se lee directo (sin copiar)

    gdf = gpd.read_file(ruta).to_crs(4326)
    try:
        geometria = gdf.geometry.union_all()      # shapely >= 2 (disuelve features)
    except AttributeError:
        geometria = gdf.geometry.unary_union      # shapely < 2
    gdf = gpd.GeoDataFrame(geometry=[geometria], crs=4326)

    roi = geemap.geopandas_to_ee(gdf).geometry()

    minx, miny, maxx, maxy = geometria.bounds
    ZONA["bbox"] = [minx, miny, maxx, maxy]
    ZONA["centro"] = [(minx + maxx) / 2, (miny + maxy) / 2]
    ZONA["zoom"] = 10
    print("AOI cargado desde:", ruta)
else:
    roi = ee.Geometry.Rectangle(ZONA["bbox"], proj="EPSG:4326", geodesic=False)

centro = ee.Geometry.Point(ZONA["centro"])
area_km2 = roi.area(maxError=1e6).divide(1e6).getInfo()

Map_aoi = geemap.Map()
Map_aoi.centerObject(roi, ZONA["zoom"])
Map_aoi.add_basemap("HYBRID")
Map_aoi.addLayer(borde_aoi(), {}, f"AOI: {ZONA['nombre']}")

# Puntos del saber local (si se cargaron en CONFIG)
for tipo, color in (("manantiales", "cyan"), ("bofedales", "magenta")):
    coords = PUNTOS_LOCALES.get(tipo, [])
    if coords:
        Map_aoi.addLayer(
            ee.Geometry.MultiPoint(coords),
            {"color": color},
            f"{tipo.capitalize()} (saber local)",
        )

print(f"Zona: {ZONA['nombre']}")
print(f"Área del AOI: {area_km2:,.1f} km²")
print(f"Extensión (bbox): {ZONA['bbox']}")
print("Puntos de saber local cargados:",
      {k: len(v) for k, v in PUNTOS_LOCALES.items()})

Map_aoi

# M1. Fisiografía: elevación, pendiente, orientación y drenaje
**Dimensiones:** *estático* (sin dimensión temporal) — pero define dónde ocurren todas las demás.

El relieve condiciona casi todo: dónde se acumula el aire frío (inversiones térmicas en valles),
qué partes son mecanizables, por dónde corre el agua y dónde se forman los bofedales (planicies).
En el Altiplano y los Valles, la elevación también explica la variación de la helada con pocos
metros de desnivel.

* **Primario:** `USGS/SRTMGL1_003` (30 m, global, ±16 m de error vertical).
* **Alternativas:** Copernicus DEM GLO30 (30 m, más reciente) o NASADEM (fusión SRTM + ICESat).
* **Drenaje:** `WWF/HydroSHEDS/15ACC` = acumulación de flujo (celdas aguas arriba); define la red
  hídrica y sirve en M9 para separar cabeceras de ríos grandes.

In [ ]:
# ============================================================
# M1.1 Elevación, pendiente y aspecto
# ============================================================

# --- Primario ---------------------------------------------------
dem = ee.Image("USGS/SRTMGL1_003").select("elevation")

# --- Alternativas (descomentar una) -----------------------------
# dem = ee.Image("COPERNICUS/DEM/GLO30").select("DEM")          # Copernicus 30 m
# dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")    # NASADEM 30 m

dem = dem.clip(roi)
relieve = ee.Terrain.products(dem)

elev = relieve.select("elevation")
pend = relieve.select("slope")
aspecto = relieve.select("aspect")

# --- Estadísticos dentro del AOI --------------------------------
stats = (
    ee.Image.cat([elev.rename("elev"), pend.rename("pend")])
    .reduceRegion(
        ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
        roi, ESCALAS["clima"], maxPixels=1e10, tileScale=TILE_SCALE,
        bestEffort=True,
    )
    .getInfo()
)
E_MIN = stats["elev_min"]
E_MAX = stats["elev_max"]
P_PROM = stats["pend_mean"]

print(f"Elevación: {E_MIN:,.0f} – {E_MAX:,.0f} msnm  (rango {E_MAX - E_MIN:,.0f} m)")
print(f"Pendiente media: {P_PROM:.1f}°")

# --- Mapa -------------------------------------------------------
vis_elev = {
    "min": E_MIN, "max": E_MAX,
    "palette": ["1a4f00", "2e7d32", "8bc34a", "fff176", "ff8f00", "8d6e63", "f5f5f5"],
}
vis_pend = {
    "min": 0, "max": 45,
    "palette": ["ffffcc", "a1dab4", "41b6c4", "2c7fb8", "253494"],
}
vis_asp = {
    "min": 0, "max": 360,
    "palette": ["ff0000", "ff8000", "ffff00", "00ff00", "00ffff", "0000ff", "8000ff", "ff0000"],
}

Map_m1 = geemap.Map()
Map_m1.centerObject(roi, ZONA["zoom"])
Map_m1.add_basemap("HYBRID")
Map_m1.addLayer(elev, vis_elev, "Elevación (msnm)")
Map_m1.addLayer(pend, vis_pend, "Pendiente (°)")
Map_m1.addLayer(aspecto, vis_asp, "Aspecto / orientación (°)", False)
Map_m1.addLayer(borde_aoi(), {}, "AOI")
Map_m1.add_colorbar(vis_elev, label="Elevación (msnm)",
                    orientation="horizontal", position="bottomright",
                    layer_name="Elevación (msnm)")
Map_m1

In [ ]:
# ============================================================
# M1.2 Restricciones del terreno: bandas de elevación, pendiente y orientación
# ============================================================

pix_km2 = ee.Image.pixelArea().divide(1e6)

# --- 1) Superficie por bandas de elevación (paso de 500 m) ------------------
PASO_M = 500
piso = int(np.floor(E_MIN / PASO_M) * PASO_M)
techo = int(np.ceil(E_MAX / PASO_M) * PASO_M)
limites = list(range(piso, techo + PASO_M, PASO_M))

bandas_elev, claves_elev = [], []
for lo, hi in zip(limites[:-1], limites[1:]):
    mascara = elev.gte(lo).And(elev.lt(hi))
    clave = f"e_{lo}_{hi}"
    bandas_elev.append(pix_km2.updateMask(mascara).rename(clave))
    claves_elev.append((clave, f"{lo}-{hi}"))

areas_elev = reducir_bandas(
    bandas_elev, [c for c, _ in claves_elev], ESCALAS["clima"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
df_elev = pd.DataFrame({
    "banda_msnm": [etiqueta for _, etiqueta in claves_elev],
    "km2": [areas_elev.get(c, 0) or 0 for c, _ in claves_elev],
})
df_elev["pct"] = 100 * df_elev["km2"] / df_elev["km2"].sum()

# --- 2) Pendiente por clases de trabajo -------------------------------------
clases_pend = {
    "p0_2": (pend.gte(0).And(pend.lt(2)), "plana 0-2"),
    "p2_5": (pend.gte(2).And(pend.lt(5)), "suave 2-5"),
    "p5_15": (pend.gte(5).And(pend.lt(15)), "media 5-15"),
    "p15_30": (pend.gte(15).And(pend.lt(30)), "fuerte 15-30"),
    "p30m": (pend.gte(30), "muy fuerte >=30"),
}
bandas_pend = [
    pix_km2.updateMask(mascara).rename(clave)
    for clave, (mascara, _) in clases_pend.items()
]
areas_pend = reducir_bandas(
    bandas_pend, [clave for clave, _ in clases_pend.items()], ESCALAS["clima"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
df_pend = pd.DataFrame({
    "clase": [etiqueta for _, (_, etiqueta) in clases_pend.items()],
    "km2": [areas_pend.get(clave, 0) or 0 for clave, _ in clases_pend.items()],
})
df_pend["pct"] = 100 * df_pend["km2"] / df_pend["km2"].sum()

# --- 3) Orientación: cara sur vs. cara norte (hemisferio sur) ---------------
cara_sur = aspecto.gte(112.5).And(aspecto.lt(247.5))
bandas_asp = [
    pix_km2.updateMask(cara_sur).rename("sur"),
    pix_km2.updateMask(cara_sur.Not()).rename("norte"),
]
areas_asp = reducir_bandas(
    bandas_asp, ["sur", "norte"], ESCALAS["clima"],
    reductor=ee.Reducer.sum(), params=REDUCIR_AREA,
)
pct_sur = 100 * (areas_asp.get("sur") or 0) / max(area_km2, 1e-9)
print(f"Área con cara sur (112–248°): {pct_sur:.1f}% (en el sur acumula más frío y humedad)")

display(df_elev.round(1))
display(df_pend.round(1))

# --- Gráficos ----------------------------------------------------------------
fig, ejes = plt.subplots(1, 2, figsize=(14, 4.5))

ejes[0].bar(df_elev["banda_msnm"], df_elev["pct"], color="#2e7d32")
ejes[0].set_title("Área por elevación")
ejes[0].set_xlabel("Banda de elevación (msnm)")
ejes[0].set_ylabel("% del área")
ejes[0].tick_params(axis="x", rotation=45)

ejes[1].bar(df_pend["clase"], df_pend["pct"], color="#1565c0")
ejes[1].set_title("Área por pendiente")
ejes[1].set_xlabel("Clase de pendiente")
ejes[1].set_ylabel("% del área")
ejes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# M1.3 Red de drenaje (acumulación de flujo)
# ============================================================

# --- Primario ---------------------------------------------------
drenaje = ee.Image("WWF/HydroSHEDS/15ACC").select("b1")

# --- Alternativas -----------------------------------------------
# drenaje = ee.Image("WWF/HydroSHEDS/30ACC").select("b1")   # ~900 m, más rápido

UMBRAL_CELDAS = 100        # >= 100 celdas acumuladas ~ cauce bien definido
rios = drenaje.gte(UMBRAL_CELDAS).selfMask().clip(roi)

Map_m1b = geemap.Map()
Map_m1b.centerObject(roi, ZONA["zoom"])
Map_m1b.add_basemap("HYBRID")
Map_m1b.addLayer(elev, vis_elev, "Elevación (msnm)", False)
Map_m1b.addLayer(rios, {"palette": ["1565c0"]},
                 f"Drenaje (>= {UMBRAL_CELDAS} celdas acumuladas)")
Map_m1b.addLayer(borde_aoi(), {}, "AOI")
Map_m1b

## Interpretación de M1
* **Elevación:** define pisos térmicos; compara el rango con los cultivos prioritarios
  (papa, quinua, caña, frutales) y con la altitud reportada por las comunidades.
* **Pendiente:** < 5° suele permitir mecanización liviana y riego por gravedad; > 15° obliga a
  terrazas/cosecha de agua y aumenta la erosión.
* **Cara sur:** en el hemisferio sur, laderas sur reciben menos radiación → suelos más fríos y
  mayor probabilidad de helada; es donde más conviene el monitoreo con estaciones CLIMBAT.
* **Drenaje:** la red define las microcuencas de intervención y los puntos de captación.

**Preguntas guía**
1. ¿La elevación de sus parcelas reportada por los agricultores cae dentro de las bandas dominantes?
2. ¿Qué porcentaje del territorio es realmente mecanizable?
3. ¿Dónde propondrían instalar una estación agroclimática con base en relieve y orientación?

## Cierre: checklist del modulo

- [ ] Parte base ejecutada (instalacion, autenticacion GEE, CONFIG, AOI del GeoJSON).
- [ ] Mapa(s) revisados contra el borde rojo del AOI (el GeoJSON siempre manda).
- [ ] Serie/tabla leidas + interpretacion contrastada con saber local (Yapuchiri/PACHASOL).
- [ ] Periodo y dataset anotados junto a cada numero (p. ej. `CHIRPS 1981-2025`).

### Como cambiar de satelite o de anos (patron en todos los modulos)

```python
FECHA_INICIO = "1996-01-01"  # en la celda CONFIG: ventana analitica
FECHA_FIN = "2026-12-31"
# --- Primario (default) ---
precip = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# --- Alternativas: comentar la primaria y descomentar la elegida ---
# precip = ee.ImageCollection("NASA/GPM_L3/IMERG_V06")
# Cada alternativa indica resolucion, periodo y para que sirve.
```
